# 06 — Results and Analysis

This notebook analyzes the final evaluated outputs created by
`05_evaluate_saved_outputs.ipynb`.

It does **not**:
- query benchmark language models,
- select automatic evaluators,
- tune semantic thresholds,
- modify the final evaluated data.

The main analyses are:

1. generation coverage,
2. answer correctness,
3. context-grounded reasoning support,
4. confidence calibration,
5. matched base-versus-variant robustness,
6. pairwise model differences on common items,
7. category / difficulty / variant-type summaries.

Because several variants come from the same base question, variant analyses
treat `base_question_id` as the clustering unit rather than assuming all
variant rows are independent.


## Setup


In [ ]:
%pip install -q pandas numpy matplotlib scikit-learn


In [ ]:
from pathlib import Path
from itertools import combinations
from datetime import datetime, timezone

import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Project location and configuration

Normally only `EVALUATION_LABEL` needs to match the label used in notebook 05.

`BOOTSTRAP_ITERATIONS` controls clustered confidence intervals.

`ECE_BINS` is used only for the secondary descriptive Expected Calibration
Error (ECE). Mean Brier score is treated as the primary calibration summary.


In [ ]:
USE_GOOGLE_DRIVE = False

DRIVE_PROJECT_PATH = Path(
    "/content/drive/MyDrive/turkish-llm-reasoning"
)

EVALUATION_LABEL = "journal_eval_v1"

BOOTSTRAP_ITERATIONS = 2000
RANDOM_SEED = 20260926
ECE_BINS = 10


In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = DRIVE_PROJECT_PATH

else:
    current = Path.cwd().resolve()
    PROJECT_ROOT = None

    for candidate in [current, *current.parents]:
        if (
            (candidate / "notebooks").exists()
            and (candidate / "src").exists()
            and (candidate / "data").exists()
        ):
            PROJECT_ROOT = candidate
            break

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Project root could not be found. "
            "Run this notebook inside the repository "
            "or enable Google Drive mode."
        )


FINAL_DIR = (
    PROJECT_ROOT
    / "results"
    / "evaluated"
    / EVALUATION_LABEL
    / "final"
)

ANALYSIS_DIR = (
    PROJECT_ROOT
    / "results"
    / "evaluated"
    / EVALUATION_LABEL
    / "analysis"
)

FIGURE_DIR = ANALYSIS_DIR / "figures"

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Final evaluation:", FINAL_DIR)
print("Analysis output:", ANALYSIS_DIR)


## Load final evaluated outputs

Notebook 06 only reads the frozen outputs produced by notebook 05.


In [ ]:
required_files = {
    "base": FINAL_DIR / "base_evaluated.csv",
    "variants": FINAL_DIR / "variants_evaluated.csv",
    "manifest": FINAL_DIR / "source_manifest.csv",
    "metadata": FINAL_DIR / "evaluation_metadata.json",
}

missing = [
    str(path)
    for path in required_files.values()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Notebook 05 full evaluation has not produced all required files:\n"
        + "\n".join(missing)
    )


base = pd.read_csv(required_files["base"])
variants = pd.read_csv(required_files["variants"])
manifest = pd.read_csv(required_files["manifest"])

with open(required_files["metadata"], "r", encoding="utf-8") as file:
    evaluation_metadata = json.load(file)


claims_path = FINAL_DIR / "claim_level_evaluation.csv"

claims = (
    pd.read_csv(claims_path)
    if claims_path.exists()
    else pd.DataFrame()
)


print("Base rows:", len(base))
print("Variant rows:", len(variants))
print("Claim rows:", len(claims))


## Integrity checks

These checks do not re-evaluate model answers. They only confirm that the
analysis input has the fields notebook 06 expects.


In [ ]:
required_common = {
    "record_id",
    "model_label",
    "model_access",
    "generation_success",
    "answer_correct",
    "end_to_end_correct",
    "confidence_normalized",
    "brier_score",
    "context_support_rate",
    "context_contradiction_rate",
    "context_nei_rate",
}

missing_base = required_common - set(base.columns)
missing_variants = (
    required_common
    | {
        "base_question_id",
        "variant_type",
        "morphological_feature",
    }
) - set(variants.columns)

if missing_base:
    raise ValueError(
        f"Base evaluated data is missing: {sorted(missing_base)}"
    )

if missing_variants:
    raise ValueError(
        f"Variant evaluated data is missing: {sorted(missing_variants)}"
    )


for name, dataframe in (
    ("base", base),
    ("variants", variants),
):
    duplicates = dataframe.duplicated(
        subset=["model_label", "record_id"]
    )

    if duplicates.any():
        raise ValueError(
            f"{name} contains duplicate model/item rows."
        )


print("Integrity checks passed.")


## Analysis helpers

Confidence intervals use a clustered bootstrap.

- Base questions: cluster = `record_id`
- Variants: cluster = `base_question_id`

For variants, this prevents multiple variants derived from one base question
from being treated as fully independent observations.

The reported estimate is the mean of cluster-level means, so each base
question receives equal weight.


In [ ]:
def cluster_mean_ci(
    dataframe,
    value_col,
    cluster_col,
    *,
    n_boot=BOOTSTRAP_ITERATIONS,
    seed=RANDOM_SEED,
):
    data = dataframe[
        [cluster_col, value_col]
    ].dropna().copy()

    if data.empty:
        return {
            "estimate": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "n_rows": 0,
            "n_clusters": 0,
        }

    cluster_means = (
        data.groupby(cluster_col)[value_col]
        .mean()
        .astype(float)
        .to_numpy()
    )

    estimate = float(cluster_means.mean())
    n_clusters = len(cluster_means)

    if n_clusters == 1:
        return {
            "estimate": estimate,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "n_rows": len(data),
            "n_clusters": 1,
        }

    rng = np.random.default_rng(seed)

    samples = rng.choice(
        cluster_means,
        size=(n_boot, n_clusters),
        replace=True,
    )

    boot_means = samples.mean(axis=1)

    return {
        "estimate": estimate,
        "ci_low": float(np.quantile(boot_means, 0.025)),
        "ci_high": float(np.quantile(boot_means, 0.975)),
        "n_rows": int(len(data)),
        "n_clusters": int(n_clusters),
    }


def grouped_cluster_summary(
    dataframe,
    value_col,
    cluster_col,
    group_cols,
):
    rows = []

    for keys, group in dataframe.groupby(
        list(group_cols),
        dropna=False,
        sort=True,
    ):
        if not isinstance(keys, tuple):
            keys = (keys,)

        result = cluster_mean_ci(
            group,
            value_col,
            cluster_col,
        )

        row = dict(zip(group_cols, keys))
        row.update(result)
        rows.append(row)

    return pd.DataFrame(rows)


def ece_score(
    correctness,
    confidence,
    *,
    bins=ECE_BINS,
):
    frame = pd.DataFrame(
        {
            "correct": correctness,
            "confidence": confidence,
        }
    ).dropna()

    if frame.empty:
        return np.nan

    edges = np.linspace(0.0, 1.0, bins + 1)

    frame["bin"] = pd.cut(
        frame["confidence"],
        bins=edges,
        include_lowest=True,
        right=True,
    )

    ece = 0.0
    total = len(frame)

    for _, group in frame.groupby(
        "bin",
        observed=False,
    ):
        if group.empty:
            continue

        weight = len(group) / total
        accuracy = group["correct"].mean()
        mean_confidence = group["confidence"].mean()

        ece += weight * abs(
            accuracy - mean_confidence
        )

    return float(ece)


def reliability_table(
    dataframe,
    *,
    bins=ECE_BINS,
):
    data = dataframe[
        [
            "answer_correct",
            "confidence_normalized",
        ]
    ].dropna().copy()

    if data.empty:
        return pd.DataFrame()

    edges = np.linspace(0.0, 1.0, bins + 1)

    data["bin"] = pd.cut(
        data["confidence_normalized"],
        bins=edges,
        include_lowest=True,
        right=True,
    )

    return (
        data.groupby("bin", observed=False)
        .agg(
            n=("answer_correct", "size"),
            mean_confidence=(
                "confidence_normalized",
                "mean",
            ),
            empirical_accuracy=(
                "answer_correct",
                "mean",
            ),
        )
        .reset_index()
    )


## Generation coverage

Generation success is reported separately from answer correctness.

This distinction matters because a failed generation is not the same event as a
successfully generated but incorrect answer.


In [ ]:
coverage = (
    pd.concat(
        [
            base.assign(input_type="base"),
            variants.assign(input_type="variants"),
        ],
        ignore_index=True,
        sort=False,
    )
    .groupby(
        [
            "model_access",
            "input_type",
            "model_label",
        ],
        dropna=False,
    )
    .agg(
        total_records=("record_id", "size"),
        successful_generations=(
            "generation_success",
            "sum",
        ),
    )
    .reset_index()
)

coverage["generation_success_rate"] = (
    coverage["successful_generations"]
    / coverage["total_records"]
)

display(coverage)

coverage.to_csv(
    ANALYSIS_DIR / "generation_coverage.csv",
    index=False,
)


## Answer correctness

Two views are kept separate:

- **Conditional accuracy:** correctness among outputs that were successfully
  generated and could be scored.
- **End-to-end accuracy:** generation failures count as unsuccessful outcomes.

For variant data, confidence intervals are clustered by `base_question_id`.


In [ ]:
def accuracy_table(dataframe, input_type):
    cluster_col = (
        "record_id"
        if input_type == "base"
        else "base_question_id"
    )

    conditional = dataframe[
        dataframe["answer_correct"].notna()
    ].copy()

    conditional_table = grouped_cluster_summary(
        conditional,
        "answer_correct",
        cluster_col,
        ["model_access", "model_label"],
    )
    conditional_table["input_type"] = input_type
    conditional_table["metric"] = "conditional_accuracy"

    e2e_table = grouped_cluster_summary(
        dataframe,
        "end_to_end_correct",
        cluster_col,
        ["model_access", "model_label"],
    )
    e2e_table["input_type"] = input_type
    e2e_table["metric"] = "end_to_end_accuracy"

    return pd.concat(
        [conditional_table, e2e_table],
        ignore_index=True,
    )


accuracy_summary = pd.concat(
    [
        accuracy_table(base, "base"),
        accuracy_table(variants, "variants"),
    ],
    ignore_index=True,
)

accuracy_summary = accuracy_summary[
    [
        "model_access",
        "model_label",
        "input_type",
        "metric",
        "estimate",
        "ci_low",
        "ci_high",
        "n_rows",
        "n_clusters",
    ]
]

display(accuracy_summary)

accuracy_summary.to_csv(
    ANALYSIS_DIR / "answer_accuracy_summary.csv",
    index=False,
)


## Context-grounded reasoning support

Reasoning is summarized at the model-output level using the claim-level NLI
results created in notebook 05.

The main descriptive quantities are kept separate:

- support rate,
- contradiction rate,
- NEI / neutral rate,
- mean entailment probability.

No weighted "overall faithfulness score" is created.


In [ ]:
reasoning_rows = []

for input_type, dataframe, cluster_col in (
    ("base", base, "record_id"),
    ("variants", variants, "base_question_id"),
):
    for metric in (
        "context_support_rate",
        "context_contradiction_rate",
        "context_nei_rate",
        "mean_context_entailment_probability",
    ):
        summary = grouped_cluster_summary(
            dataframe[
                dataframe[metric].notna()
            ],
            metric,
            cluster_col,
            ["model_access", "model_label"],
        )

        summary["input_type"] = input_type
        summary["metric"] = metric
        reasoning_rows.append(summary)


reasoning_summary = pd.concat(
    reasoning_rows,
    ignore_index=True,
)

reasoning_summary = reasoning_summary[
    [
        "model_access",
        "model_label",
        "input_type",
        "metric",
        "estimate",
        "ci_low",
        "ci_high",
        "n_rows",
        "n_clusters",
    ]
]

display(reasoning_summary)

reasoning_summary.to_csv(
    ANALYSIS_DIR / "reasoning_support_summary.csv",
    index=False,
)


## Confidence calibration

Mean Brier score is the primary calibration summary.

A 10-bin ECE and reliability tables are also produced as secondary,
bin-dependent descriptive summaries.


In [ ]:
calibration_rows = []

for input_type, dataframe, cluster_col in (
    ("base", base, "record_id"),
    ("variants", variants, "base_question_id"),
):
    for (model_access, model_label), group in dataframe.groupby(
        ["model_access", "model_label"],
        dropna=False,
        sort=True,
    ):
        scored = group[
            group["answer_correct"].notna()
            & group["confidence_normalized"].notna()
        ].copy()

        brier = cluster_mean_ci(
            scored,
            "brier_score",
            cluster_col,
        )

        calibration_rows.append(
            {
                "model_access": model_access,
                "model_label": model_label,
                "input_type": input_type,
                "mean_brier": brier["estimate"],
                "brier_ci_low": brier["ci_low"],
                "brier_ci_high": brier["ci_high"],
                "ece_10_bin": ece_score(
                    scored["answer_correct"],
                    scored["confidence_normalized"],
                    bins=ECE_BINS,
                ),
                "mean_confidence": (
                    scored["confidence_normalized"].mean()
                    if len(scored)
                    else np.nan
                ),
                "empirical_accuracy": (
                    scored["answer_correct"].mean()
                    if len(scored)
                    else np.nan
                ),
                "n_scored": len(scored),
                "n_clusters": brier["n_clusters"],
            }
        )


calibration_summary = pd.DataFrame(
    calibration_rows
)

display(calibration_summary)

calibration_summary.to_csv(
    ANALYSIS_DIR / "calibration_summary.csv",
    index=False,
)


### Reliability plots

One figure is saved for each model and input type.


In [ ]:
for input_type, dataframe in (
    ("base", base),
    ("variants", variants),
):
    for model_label, group in dataframe.groupby(
        "model_label",
        dropna=False,
        sort=True,
    ):
        table = reliability_table(
            group,
            bins=ECE_BINS,
        )

        if table.empty:
            continue

        figure, axis = plt.subplots(
            figsize=(6, 5)
        )

        axis.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
        )

        axis.plot(
            table["mean_confidence"],
            table["empirical_accuracy"],
            marker="o",
        )

        axis.set_xlim(0, 1)
        axis.set_ylim(0, 1)
        axis.set_xlabel("Mean reported confidence")
        axis.set_ylabel("Empirical accuracy")
        axis.set_title(
            f"Reliability — {model_label} — {input_type}"
        )

        figure.tight_layout()

        safe_name = (
            str(model_label)
            .replace("/", "_")
            .replace(" ", "_")
        )

        figure.savefig(
            FIGURE_DIR
            / f"reliability__{safe_name}__{input_type}.png",
            dpi=200,
            bbox_inches="tight",
        )

        plt.show()
        plt.close(figure)


## Matched base-versus-variant robustness

Variant rows are matched to the corresponding base question for the same
model.

Only models with both a base result and the corresponding variant result can
contribute to a matched comparison.

The primary robustness contrast below uses end-to-end correctness:

`variant correctness - base correctness`

A negative value means lower correctness on the variants; a positive value
means higher correctness on the variants.

Confidence intervals are clustered at `base_question_id`.


In [ ]:
base_lookup = base[
    [
        "model_label",
        "record_id",
        "end_to_end_correct",
        "answer_correct",
    ]
].rename(
    columns={
        "record_id": "base_question_id",
        "end_to_end_correct": "base_end_to_end_correct",
        "answer_correct": "base_answer_correct",
    }
)

matched = variants.merge(
    base_lookup,
    on=["model_label", "base_question_id"],
    how="inner",
    validate="many_to_one",
)

matched["e2e_delta"] = (
    matched["end_to_end_correct"].astype(float)
    - matched["base_end_to_end_correct"].astype(float)
)

both_scored = (
    matched["answer_correct"].notna()
    & matched["base_answer_correct"].notna()
)

matched["conditional_delta"] = np.nan

matched.loc[
    both_scored,
    "conditional_delta",
] = (
    matched.loc[
        both_scored,
        "answer_correct",
    ].astype(float)
    -
    matched.loc[
        both_scored,
        "base_answer_correct",
    ].astype(float)
)


robustness_rows = []

for model_label, group in matched.groupby(
    "model_label",
    dropna=False,
    sort=True,
):
    e2e = cluster_mean_ci(
        group,
        "e2e_delta",
        "base_question_id",
    )

    conditional = cluster_mean_ci(
        group,
        "conditional_delta",
        "base_question_id",
    )

    robustness_rows.append(
        {
            "model_label": model_label,
            "matched_variant_rows": len(group),
            "matched_base_questions": (
                group["base_question_id"].nunique()
            ),
            "e2e_delta": e2e["estimate"],
            "e2e_ci_low": e2e["ci_low"],
            "e2e_ci_high": e2e["ci_high"],
            "conditional_delta": conditional["estimate"],
            "conditional_ci_low": conditional["ci_low"],
            "conditional_ci_high": conditional["ci_high"],
        }
    )


robustness_summary = pd.DataFrame(
    robustness_rows
)

display(robustness_summary)

robustness_summary.to_csv(
    ANALYSIS_DIR / "matched_base_variant_robustness.csv",
    index=False,
)


### Variant-type robustness

This table keeps variant types separate while still clustering uncertainty by
the original base question.


In [ ]:
variant_type_rows = []

for (model_label, variant_type), group in matched.groupby(
    ["model_label", "variant_type"],
    dropna=False,
    sort=True,
):
    result = cluster_mean_ci(
        group,
        "e2e_delta",
        "base_question_id",
    )

    variant_type_rows.append(
        {
            "model_label": model_label,
            "variant_type": variant_type,
            "estimate": result["estimate"],
            "ci_low": result["ci_low"],
            "ci_high": result["ci_high"],
            "n_variant_rows": result["n_rows"],
            "n_base_questions": result["n_clusters"],
        }
    )


variant_type_robustness = pd.DataFrame(
    variant_type_rows
)

display(variant_type_robustness)

variant_type_robustness.to_csv(
    ANALYSIS_DIR / "variant_type_robustness.csv",
    index=False,
)


## Pairwise model differences on common items

These comparisons use only items answered by both models.

No model is assigned an overall rank.

For each alphabetically ordered model pair, the reported contrast is:

`model_A end-to-end correctness - model_B end-to-end correctness`

Base comparisons are paired by `record_id`.

Variant comparisons are paired by variant `record_id`, with confidence
intervals clustered by `base_question_id`.


In [ ]:
def pairwise_model_differences(
    dataframe,
    *,
    item_col,
    cluster_col,
    value_col="end_to_end_correct",
):
    models = sorted(
        dataframe["model_label"]
        .dropna()
        .astype(str)
        .unique()
    )

    rows = []

    for model_a, model_b in combinations(
        models,
        2,
    ):
        left_source = dataframe[
            dataframe["model_label"].astype(str) == model_a
        ]

        right_source = dataframe[
            dataframe["model_label"].astype(str) == model_b
        ]

        # `item_col` and `cluster_col` are both `record_id` for the
        # base dataset. Build the pairing tables explicitly so that we
        # never create duplicate column names in that case.
        left = left_source[
            [item_col, value_col]
        ].copy()

        right = right_source[
            [item_col, value_col]
        ].copy()

        left["cluster_a"] = (
            left_source[cluster_col]
            .astype(str)
            .to_numpy()
        )

        right["cluster_b"] = (
            right_source[cluster_col]
            .astype(str)
            .to_numpy()
        )

        left = left.rename(
            columns={
                value_col: "value_a",
            }
        )

        right = right.rename(
            columns={
                value_col: "value_b",
            }
        )

        paired = left.merge(
            right,
            on=item_col,
            how="inner",
            validate="one_to_one",
        )

        if paired.empty:
            continue

        if not (
            paired["cluster_a"].astype(str)
            == paired["cluster_b"].astype(str)
        ).all():
            raise ValueError(
                "Paired model rows disagree on cluster identity."
            )

        paired["delta"] = (
            paired["value_a"].astype(float)
            - paired["value_b"].astype(float)
        )

        result = cluster_mean_ci(
            paired,
            "delta",
            "cluster_a",
        )

        rows.append(
            {
                "model_a": model_a,
                "model_b": model_b,
                "contrast": "model_a_minus_model_b",
                "estimate": result["estimate"],
                "ci_low": result["ci_low"],
                "ci_high": result["ci_high"],
                "n_common_items": len(paired),
                "n_clusters": result["n_clusters"],
            }
        )

    return pd.DataFrame(rows)


base_pairwise = pairwise_model_differences(
    base,
    item_col="record_id",
    cluster_col="record_id",
)

variant_pairwise = pairwise_model_differences(
    variants,
    item_col="record_id",
    cluster_col="base_question_id",
)


display(base_pairwise)
display(variant_pairwise)


base_pairwise.to_csv(
    ANALYSIS_DIR / "pairwise_models_base.csv",
    index=False,
)

variant_pairwise.to_csv(
    ANALYSIS_DIR / "pairwise_models_variants.csv",
    index=False,
)


## Category, difficulty and variant-type summaries

These are descriptive subgroup analyses. They are not used to choose or tune
the evaluators.


In [ ]:
base_subgroups = (
    base.groupby(
        [
            "model_label",
            "category",
            "difficulty",
        ],
        dropna=False,
    )
    .agg(
        n=("record_id", "size"),
        generation_success_rate=(
            "generation_success",
            "mean",
        ),
        end_to_end_accuracy=(
            "end_to_end_correct",
            "mean",
        ),
    )
    .reset_index()
)

variant_subgroups = (
    variants.groupby(
        [
            "model_label",
            "category",
            "difficulty",
            "variant_type",
        ],
        dropna=False,
    )
    .agg(
        n=("record_id", "size"),
        base_questions=(
            "base_question_id",
            "nunique",
        ),
        generation_success_rate=(
            "generation_success",
            "mean",
        ),
        end_to_end_accuracy=(
            "end_to_end_correct",
            "mean",
        ),
    )
    .reset_index()
)


display(base_subgroups.head(30))
display(variant_subgroups.head(30))


base_subgroups.to_csv(
    ANALYSIS_DIR / "base_subgroup_summary.csv",
    index=False,
)

variant_subgroups.to_csv(
    ANALYSIS_DIR / "variant_subgroup_summary.csv",
    index=False,
)


## Analysis metadata and output inventory

This final cell records the analysis settings used for the saved tables and
figures.


In [ ]:
analysis_metadata = {
    "evaluation_label": EVALUATION_LABEL,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "random_seed": RANDOM_SEED,
    "cluster_definition": {
        "base": "record_id",
        "variants": "base_question_id",
    },
    "calibration": {
        "primary_summary": "mean_brier_score",
        "secondary_ece_bins": ECE_BINS,
    },
    "robustness_primary_contrast": (
        "variant_end_to_end_correct_minus_base_end_to_end_correct"
    ),
    "pairwise_model_contrast": (
        "alphabetically_ordered_model_a_minus_model_b_on_common_items"
    ),
    "notes": [
        "Generation success is reported separately from conditional accuracy.",
        "Variant confidence intervals are clustered by base_question_id.",
        "No automatic evaluator is selected or tuned in notebook 06.",
        "No overall weighted faithfulness score is constructed.",
    ],
}

with open(
    ANALYSIS_DIR / "analysis_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        analysis_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )


print("Analysis outputs:")

for path in sorted(ANALYSIS_DIR.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(PROJECT_ROOT))
